# Lesson 2: Calculus, Chain Rule, Probability & Optimization

**Runtime Tier**: Google Colab Free Tier **CPU** (`numpy`, `matplotlib`)

Build numerical gradient checks, a reverse-mode `Value` autograd node, stable softmax, and Bayes updates.


In [ ]:
import numpy as np

def f(x):
    return (3.0 * x + 1.0) ** 2

x = 2.0
h = 1e-5
num_grad = (f(x + h) - f(x - h)) / (2 * h)
ana_grad = 6.0 * (3.0 * x + 1.0)
print(f'Numerical: {num_grad:.6f}, Analytical: {ana_grad:.6f}')


In [ ]:
# Minimal Reverse-Mode Autograd Value Node
class Value:
    def __init__(self, data, _children=()):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

a, b = Value(2.0), Value(3.0)
c = (a * b + a) * Value(4.0)
c.backward()
print('dc/da:', a.grad, 'dc/db:', b.grad)


In [ ]:
# Stable Softmax, Cross-Entropy & Bayes Update
logits = np.array([1000.0, 1002.0, 999.0])
probs = np.exp(logits - logits.max())
probs /= probs.sum()
y_true = np.array([0.0, 1.0, 0.0])
ce = -np.sum(y_true * np.log(probs + 1e-12))
print('Probs:', np.round(probs, 4), 'CE:', round(float(ce), 4))

p_d, p_pos_d, p_pos_h = 0.01, 0.95, 0.05
posterior = (p_pos_d * p_d) / (p_pos_d * p_d + p_pos_h * (1 - p_d))
print('Bayes Posterior P(Disease|+):', round(posterior, 4))
